# Yellow Taxi Trip Reports

In [1]:
import pandas as pd

In [2]:
yellow_df = pd.read_parquet("data/yellow_tripdata_2026-01.parquet")

## Exploring the data
We check for any duplicate data.

In [3]:
yellow_df[yellow_df.duplicated(subset=['tpep_pickup_datetime', 'tpep_dropoff_datetime'], keep=False)]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
74,2,2026-01-01 00:54:05,2026-01-01 00:56:36,1.0,0.63,1.0,N,143,143,4,-5.10,-1.0,-0.5,0.0,0.0,-1.0,-10.10,-2.5,0.0,0.00
75,2,2026-01-01 00:54:05,2026-01-01 00:56:36,1.0,0.63,1.0,N,143,143,4,5.10,1.0,0.5,0.0,0.0,1.0,10.10,2.5,0.0,0.00
248,2,2026-01-01 00:43:29,2026-01-01 01:09:26,1.0,1.92,1.0,N,170,79,4,-22.60,-1.0,-0.5,0.0,0.0,-1.0,-28.35,-2.5,0.0,-0.75
249,2,2026-01-01 00:43:29,2026-01-01 01:09:26,1.0,1.92,1.0,N,170,79,4,22.60,1.0,0.5,0.0,0.0,1.0,28.35,2.5,0.0,0.75
368,2,2026-01-01 00:37:39,2026-01-01 00:38:17,1.0,0.02,1.0,N,261,261,4,-3.00,-1.0,-0.5,0.0,0.0,-1.0,-8.75,-2.5,0.0,-0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3722452,1,2026-01-31 23:59:41,2026-02-01 00:11:53,NaN,1.10,NaN,NaN,107,161,0,17.85,0.0,0.5,0.0,0.0,1.0,40.66,NaN,NaN,0.75
3723020,2,2026-01-31 23:01:20,2026-01-31 23:07:31,NaN,1.06,NaN,NaN,142,237,0,11.20,0.0,0.5,0.0,0.0,1.0,15.95,NaN,NaN,0.75
3723313,1,2026-01-31 23:02:33,2026-01-31 23:10:50,NaN,1.20,NaN,NaN,125,158,0,13.87,0.0,0.5,0.0,0.0,1.0,33.87,NaN,NaN,0.75
3723586,2,2026-01-31 23:55:45,2026-02-01 00:03:55,NaN,2.34,NaN,NaN,33,148,0,29.87,0.0,0.5,0.0,0.0,1.0,34.62,NaN,NaN,0.75


We remove the duplicated data

In [4]:
yellow_df = yellow_df.drop_duplicates().reset_index(drop=True)
yellow_df['trip_id'] = yellow_df.index

We change the data type of `passenger_count`.

In [5]:
yellow_df["passenger_count"] = pd.to_numeric(yellow_df["passenger_count"], errors="coerce").astype("Int32")

In [6]:
yellow_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_id
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1,0.97,1.0,N,239,238,1,...,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00,0
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0,0.90,1.0,N,163,162,2,...,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75,1
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0,1.40,1.0,N,43,237,1,...,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75,2
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,1.0,N,142,209,1,...,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75,3
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0,2.16,1.0,N,88,144,1,...,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75,4


## Vendor table

In [7]:
vendor_name = {
    1: 'Creative Mobile Technologies, LLC',
    2: 'Curb Mobility, LLC',
    6: 'Myle Technologies Inc',
    7: 'Helix'
}

In [8]:
vendor_ids = yellow_df[['VendorID']].drop_duplicates().reset_index(drop=True)
vendor_ids['vendor_name'] = vendor_ids['VendorID'].map(vendor_name)

In [9]:
vendor_ids.head()

,VendorID,vendor_name
0,2,"Curb Mobility, LLC"
1,1,"Creative Mobile Technologies, LLC"
2,7,Helix
3,6,Myle Technologies Inc


## Datetime table

Datetime table with unique values.

In [10]:
unique_datetimes = pd.concat([
    yellow_df['tpep_pickup_datetime'],
    yellow_df['tpep_dropoff_datetime']
]).drop_duplicates().reset_index(drop=True)

In [11]:
datetimes = pd.DataFrame({'full_datetime': unique_datetimes})

In [12]:
datetimes['datetimeID'] = datetimes.index
datetimes['hour'] = datetimes['full_datetime'].dt.hour
datetimes['day'] = datetimes['full_datetime'].dt.day
datetimes['month'] = datetimes['full_datetime'].dt.month
datetimes['year'] = datetimes['full_datetime'].dt.year
datetimes['weekday'] = datetimes['full_datetime'].dt.day_name()

In [13]:
datetimes.head()

,full_datetime,datetimeID,hour,day,month,year,weekday
0,2026-01-01 00:54:04,0,0,1,1,2026,Thursday
1,2026-01-01 00:34:04,1,0,1,1,2026,Thursday
2,2026-01-01 00:57:06,2,0,1,1,2026,Thursday
3,2026-01-01 00:15:22,3,0,1,1,2026,Thursday
4,2026-01-01 00:27:13,4,0,1,1,2026,Thursday


## RatecodeID table

Dimensional table with the different rate codes. 

In [14]:
rate_code_desc = {
    1: 'Standard rate',
    2: 'JFK',
    3: 'Newark',
    4: 'Nassau or Westchester',
    5: 'Negotiated fare',
    6: 'Group ride',
    # 99: 'Null/unknown'
}

Since the `RatecodeID` in the data dictionary are integers, we must first change the data type of this column in `yellow_df`.

In [15]:
yellow_df['RatecodeID'].astype("Int32")

0             1
1             1
2             1
3             1
4             1
           ... 
3724884    <NA>
3724885    <NA>
3724886    <NA>
3724887    <NA>
3724888    <NA>
Name: RatecodeID, Length: 3724889, dtype: Int32

In [16]:
yellow_df["RatecodeID"] = pd.to_numeric(yellow_df["RatecodeID"], errors="coerce").astype("Int32")

In [17]:
yellow_df.dtypes

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                   Int32
trip_distance                   float64
RatecodeID                        Int32
store_and_fwd_flag                  str
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
trip_id                           int64
dtype: object

In [18]:
rate_codes = yellow_df[['RatecodeID']].drop_duplicates().reset_index(drop=True)
rate_codes['rate_code_desc'] = rate_codes['RatecodeID'].map(rate_code_desc)

In [19]:
rate_codes.head()

,RatecodeID,rate_code_desc
0,1,Standard rate
1,4,Nassau or Westchester
2,2,JFK
3,5,Negotiated fare
4,99,NaN


## LocationID table

Table with the pick up and drop off locations given the zone codes.

In [20]:
zones = pd.read_csv("data/taxi_zone_lookup.csv")
zones.dtypes

LocationID      int64
Borough           str
Zone              str
service_zone      str
dtype: object

In [21]:
zones

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
...,...,...,...,...
260,261,Manhattan,World Trade Center,Yellow Zone
261,262,Manhattan,Yorkville East,Yellow Zone
262,263,Manhattan,Yorkville West,Yellow Zone
263,264,Unknown,NaN,NaN


First, we change `PULocationID` and `DOLocationID` to `Int32`.

In [22]:
yellow_df["PULocationID"] = pd.to_numeric(yellow_df["PULocationID"], errors="coerce").astype("Int32")
yellow_df["DOLocationID"] = pd.to_numeric(yellow_df["DOLocationID"], errors="coerce").astype("Int32")

In [23]:
yellow_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_id
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1,0.97,1,N,239,238,1,...,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00,0
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0,0.90,1,N,163,162,2,...,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75,1
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0,1.40,1,N,43,237,1,...,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75,2
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,1,N,142,209,1,...,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75,3
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0,2.16,1,N,88,144,1,...,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75,4


### Pick up table

In [24]:
pickup_location = yellow_df[['PULocationID']].copy()
pickup_location.head()

,PULocationID
0,239
1,163
2,43
3,142
4,88


In [25]:
pickup_location = pd.merge(pickup_location, zones, how='inner', left_on='PULocationID', right_on='LocationID').drop(columns=['LocationID'])
pickup_location.head()

,PULocationID,Borough,Zone,service_zone
0,239,Manhattan,Upper West Side South,Yellow Zone
1,163,Manhattan,Midtown North,Yellow Zone
2,43,Manhattan,Central Park,Yellow Zone
3,142,Manhattan,Lincoln Square East,Yellow Zone
4,88,Manhattan,Financial District South,Yellow Zone


### Drop off table

In [26]:
dropoff_location = yellow_df[['DOLocationID']].copy()
dropoff_location.head()

,DOLocationID
0,238
1,162
2,237
3,209
4,144


In [27]:
dropoff_location = pd.merge(dropoff_location, zones, how='inner', left_on='DOLocationID', right_on='LocationID').drop(columns=['LocationID'])
dropoff_location.head()

,DOLocationID,Borough,Zone,service_zone
0,238,Manhattan,Upper West Side North,Yellow Zone
1,162,Manhattan,Midtown East,Yellow Zone
2,237,Manhattan,Upper East Side South,Yellow Zone
3,209,Manhattan,Seaport,Yellow Zone
4,144,Manhattan,Little Italy/NoLiTa,Yellow Zone


## Payment table

In [28]:
payment_type_name = {
    0: 'Flex Fare trip',
    1: 'Credit card',
    2: 'Cash',
    3: 'No charge',
    4: 'Dispute',
    5: 'Unknown',
    6: 'Voided trip'
}

In [29]:
payment_type = yellow_df[['payment_type']].drop_duplicates().reset_index(drop=True)
payment_type['payment_name'] = payment_type['payment_type'].map(payment_type_name)
payment_type.head()

,payment_type,payment_name
0,1,Credit card
1,2,Cash
2,4,Dispute
3,3,No charge
4,0,Flex Fare trip


## Yellow Taxi Trip Records table

In [30]:
yellow_df = yellow_df.merge(
    datetimes[['full_datetime', 'datetimeID']],
    left_on='tpep_pickup_datetime',
    right_on='full_datetime',
    how='left'
).rename(columns={'datetimeID': 'pickup_datetime_id'}) \
    .drop(columns=['full_datetime'])

yellow_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_id,pickup_datetime_id
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1,0.97,1,N,239,238,1,...,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00,0,0
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0,0.90,1,N,163,162,2,...,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75,1,1
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0,1.40,1,N,43,237,1,...,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75,2,2
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,1,N,142,209,1,...,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75,3,3
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0,2.16,1,N,88,144,1,...,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75,4,4


In [31]:
yellow_df = yellow_df.merge(
    datetimes[['full_datetime', 'datetimeID']],
    left_on='tpep_dropoff_datetime',
    right_on='full_datetime',
    how='left'
).rename(columns={'datetimeID': 'dropoff_datetime_id'}) \
    .drop(columns=['full_datetime'])

yellow_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_id,pickup_datetime_id,dropoff_datetime_id
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1,0.97,1,N,239,238,1,...,3.66,0.0,1.0,15.86,2.5,0.0,0.00,0,0,1453430
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0,0.90,1,N,163,162,2,...,0.00,0.0,1.0,13.65,2.5,0.0,0.75,1,1,1270
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0,1.40,1,N,43,237,1,...,2.50,0.0,1.0,18.95,2.5,0.0,0.75,2,2,4483
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,1,N,142,209,1,...,11.11,0.0,1.0,55.56,2.5,0.0,0.75,3,3,823
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0,2.16,1,N,88,144,1,...,3.85,0.0,1.0,23.10,2.5,0.0,0.75,4,4,1757191


In [32]:
yellow_taxis_records = yellow_df \
    .merge(vendor_ids, on='VendorID', how='left') \
    .merge(rate_codes, on='RatecodeID', how='left') \
    .merge(payment_type, on='payment_type', how='left') \
    .merge(zones, left_on='PULocationID', right_on='LocationID', how='left') \
    .rename(columns={'Borough': 'PU_Borough', 'Zone': 'PU_Zone', 'service_zone': 'PU_service_zone'}) \
    .drop(columns=['LocationID']) \
    .merge(zones, left_on='DOLocationID', right_on='LocationID', how='left') \
    .rename(columns={'Borough': 'DO_Borough', 'Zone': 'DO_Zone', 'service_zone': 'DO_service_zone'}) \
    .drop(columns=['LocationID'])

In [33]:
yellow_taxis_records.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee', 'trip_id', 'pickup_datetime_id',
       'dropoff_datetime_id', 'vendor_name', 'rate_code_desc', 'payment_name',
       'PU_Borough', 'PU_Zone', 'PU_service_zone', 'DO_Borough', 'DO_Zone',
       'DO_service_zone'],
      dtype='str')

In [34]:
yellow_taxis_records.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,dropoff_datetime_id,vendor_name,rate_code_desc,payment_name,PU_Borough,PU_Zone,PU_service_zone,DO_Borough,DO_Zone,DO_service_zone
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1,0.97,1,N,239,238,1,...,1453430,"Curb Mobility, LLC",Standard rate,Credit card,Manhattan,Upper West Side South,Yellow Zone,Manhattan,Upper West Side North,Yellow Zone
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0,0.90,1,N,163,162,2,...,1270,"Creative Mobile Technologies, LLC",Standard rate,Cash,Manhattan,Midtown North,Yellow Zone,Manhattan,Midtown East,Yellow Zone
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0,1.40,1,N,43,237,1,...,4483,"Creative Mobile Technologies, LLC",Standard rate,Credit card,Manhattan,Central Park,Yellow Zone,Manhattan,Upper East Side South,Yellow Zone
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,1,N,142,209,1,...,823,"Curb Mobility, LLC",Standard rate,Credit card,Manhattan,Lincoln Square East,Yellow Zone,Manhattan,Seaport,Yellow Zone
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0,2.16,1,N,88,144,1,...,1757191,"Curb Mobility, LLC",Standard rate,Credit card,Manhattan,Financial District South,Yellow Zone,Manhattan,Little Italy/NoLiTa,Yellow Zone
